In [1]:
from nltk.tokenize import sent_tokenize
import pandas as pd

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, logging
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences, get_older_labelled_data
from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.utils import prodigy_data_utils as pdu

2024-07-03 11:09:15,014 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [2]:
def match_spans_to_sentences(df):
    for idx, row in df.iterrows():
        span = row['span_sents']
        if isinstance(span, str):
            for sent in row['sentences']:
                if (span in sent) | (span==sent):
                    # logging.info(f"'''{span}''' is in {sent}")
                    df.at[idx, 'true_sentence'] = sent
        else:
            df.at[idx, 'true_sentence'] = None
    return df

def get_negative_example_sentences(df, id):
    subset = df[df['id'] == id]
    sentences = set(subset['sentences'].iloc[0])
    sentences_w_spans = set(subset['true_sentence'].unique())
    neg_sentences = sentences - sentences_w_spans
    return neg_sentences

In [3]:
labelled_sents = get_labelled_job_sentences()

2024-07-03 11:09:18,583 - dap_job_quality - INFO - File job_quality/prodigy/labelled_data/job_sentences_labelled_20240528.jsonl downloaded from open-jobs-lake to /Users/rosie.oxbury/Documents/git_repos/dap_job_quality/inputs/labelled/job_sentences_labelled_20240528.jsonl


In [4]:
len(labelled_sents)

309

In [5]:
final_labelled_ids = set([sent['meta']['id'] for sent in labelled_sents])

In [6]:
early_data = get_older_labelled_data()[10:]

2024-07-03 11:09:19,314 - dap_job_quality - INFO - File job_quality/prodigy/binary_classifier_labelled_data/20240416/job_sentences_labelled_20240416.jsonl downloaded from open-jobs-lake to /Users/rosie.oxbury/Documents/git_repos/dap_job_quality/inputs/labelled/job_sentences_labelled_20240416.jsonl


In [7]:
early_ids = set([record['id'] for record in early_data])

In [8]:
# check that no data is duplicated
common_ids = final_labelled_ids.intersection(early_ids)
common_ids

set()

In [9]:
early_data_spans = pdu.get_spans_and_sentences(early_data, chunks=False)

In [10]:
early_data_df = pd.DataFrame()

for id, val in early_data_spans.items():
    temp_df = pd.DataFrame(val)
    temp_df['id'] = id
    early_data_df = pd.concat([early_data_df, temp_df])
    
early_data_df = early_data_df.reset_index(drop=True)

early_data_df['span_sents'] = early_data_df['span'].apply(lambda x: sent_tokenize(x))
early_data_df = early_data_df.explode('span_sents')
early_data_df['sentences'] = early_data_df['text'].apply(lambda x: sent_tokenize(x))

early_data_df = early_data_df.reset_index(drop=True)

In [11]:
early_data_df = match_spans_to_sentences(early_data_df)

In [12]:
labelled_spans = pdu.get_spans_and_sentences(labelled_sents, chunks=True)

In [13]:
labelled_spans_df = pd.DataFrame()

for id, val in labelled_spans.items():
    for chunk, nested_val in val.items():
        temp_df = pd.DataFrame(nested_val)
        temp_df['id'] = id
        temp_df['chunk'] = chunk
        labelled_spans_df = pd.concat([labelled_spans_df, temp_df])
    
labelled_spans_df = labelled_spans_df.reset_index(drop=True)

In [14]:

labelled_spans_df['span_sents'] = labelled_spans_df['span'].apply(lambda x: sent_tokenize(x))
labelled_spans_df = labelled_spans_df.explode('span_sents')
labelled_spans_df['sentences'] = labelled_spans_df['text'].apply(lambda x: sent_tokenize(x))

labelled_spans_df = match_spans_to_sentences(labelled_spans_df)

In [15]:
labelled_spans_df[['id', 'text', 'label','sentences', 'span_sents', 'true_sentence']]

,id,text,label,sentences,span_sents,true_sentence
0,48347582,"Here at Avant Homes, we look to constantly cha...",none,"[Here at Avant Homes, we look to constantly ch...",NaN,None
1,48347582,"The Role Are you organised, multiskilled in pr...",none,"[The Role Are you organised, multiskilled in p...",NaN,None
2,48347582,Effectively communicate with customers with re...,none,[Effectively communicate with customers with r...,NaN,None
3,48347582,Advise the company as to recurring causes of d...,none,[Advise the company as to recurring causes of ...,NaN,None
4,48347582,Good Product Knowledge. Knowledge of Customer ...,none,"[Good Product Knowledge., Knowledge of Custome...",NaN,None
...,...,...,...,...,...,...
583,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Excellent Bonus Structure.,Excellent Bonus Structure.
584,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Career Development Opportunities.,Career Development Opportunities.
585,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Company Discount,Company Discount.
586,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Childcare Vouchers,Childcare Vouchers


In [16]:
early_data_df = early_data_df[early_data_df["span"] != ""]
labelled_spans_df = labelled_spans_df[labelled_spans_df["label"] != "none"]

In [17]:
all_labelled_data = pd.concat([early_data_df[['id', 'text', 'label','sentences', 'span_sents', 'true_sentence']], labelled_spans_df[['id', 'text', 'label','sentences', 'span_sents', 'true_sentence']]]).reset_index(drop=True)
all_labelled_data

,id,text,label,sentences,span_sents,true_sentence
0,45332364,Occupational Therapy Technician Team Leader Se...,benefit,[Occupational Therapy Technician Team Leader S...,"£25,103.32pa","Salary £25,103.32pa."
1,45332364,Occupational Therapy Technician Team Leader Se...,benefit,[Occupational Therapy Technician Team Leader S...,Hours 38 per week,Hours 38 per week Benefits Extensive access...
2,45332364,Occupational Therapy Technician Team Leader Se...,benefit,[Occupational Therapy Technician Team Leader S...,Extensive access and support to recognised qua...,Hours 38 per week Benefits Extensive access...
3,45332364,Occupational Therapy Technician Team Leader Se...,benefit,[Occupational Therapy Technician Team Leader S...,Basic entitlement is 30.0 days (pro rata for h...,Basic entitlement is 30.0 days (pro rata for h...
4,45332364,Occupational Therapy Technician Team Leader Se...,benefit,[Occupational Therapy Technician Team Leader S...,This is inclusive of Bank Holidays.,This is inclusive of Bank Holidays.
...,...,...,...,...,...,...
634,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Excellent Bonus Structure.,Excellent Bonus Structure.
635,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Career Development Opportunities.,Career Development Opportunities.
636,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Company Discount,Company Discount.
637,46827704,Excellent Bonus Structure. Career Development ...,benefit,"[Excellent Bonus Structure., Career Developmen...",Childcare Vouchers,Childcare Vouchers


In [18]:
negative_sentences = {}

for id in all_labelled_data['id'].unique():
    neg_sentences = get_negative_example_sentences(all_labelled_data, id)
    negative_sentences[id] = list(neg_sentences)

In [19]:
neg_sent_df = pd.DataFrame([(k, sentence) for k, sentences in negative_sentences.items() for sentence in sentences], columns=['id', 'sentence'])
neg_sent_df

,id,sentence
0,45332364,"To work with the people, we support and keywor..."
1,45332364,Participate in clinical governance and quality...
2,45332364,Experience of working in an occupational thera...
3,45332364,"Understanding, ability and willingness to work..."
4,45332364,QUALITY ASSURANCE and DEVELOPMENTTo ensure app...
...,...,...
841,43791725,Room and building acoustics.
842,43791725,Professional development.
843,46827704,Customer satisfaction is of paramount importa...
844,46827704,Our Autocentre Managers are our role models wh...


In [20]:
# hard-coding false negatives here, rather than removing manually - just so that there's a record of what was removed
false_negatives = ["Basic entitlement is 30.0 days (pro rata for hours worked).",
                   "Location  Bromley.",
                   "We also offer an i. Pad if you refer a new client to us and we recruit for them.",
                   "Overtime rates",
                   "7 days on call",
                   "Monday start of shift to handover the following Monday start of shift.",
                   "You will be given a training program when you start to help you integrate into the team and help you get up to date with their technologies as well as progression routes.",
                   "Work between 8.30am and 3.30pm",
                   "Salary £22,000 - 26,000p a",
                   "£250 bonus",
                   "You can discuss your preferred working hours options at interview.",
                   "A full shift would be 9-5 but there are variations of hours on offer.",
                   "Recommend a friend",
                   "6-9 calls per day dependant on area size.",
                   "They have just moved to some fantastic brand-new offices too based in South Cerney.",
                   "We are also proud to have been named a 'Top Employer' for 5 consecutive years.",
                   "You can earn up to £31,500 p a including regular overtime and bonuses.",
                   "Salary  £50,000 - £60,000",
                   "You will receive £250 for every candidate we place in permanent employment who has been recommended by you.",
                   "relocation package",
                   "This gives Colleagues and their family access to 24 7 365 support for a whole range of issues including physical, mental and financial issues.",
                   "Refer a Friend",
                   "one of the best companies to work for",
                   "has topped the UK 'Best Companies to Work For' lists for 15 years",
                   "Join one of the world's best employers!",
                   "Our client are looking for a Security Cleared Pharmacy Technician to join their team on a locum basis starting as soon as possible on an ongoing basis.",
                   "monthly commission structure",
                   "with the chance of full-time job",
                   "4 Nights out a week and can sometimes have run ins on a Saturday.",
                   "Hours of work- 9-5Contract- temporary."
                   ]

In [21]:
mask = neg_sent_df['sentence'].apply(lambda x: not any(sub in x for sub in false_negatives))

# Apply the mask to filter the DataFrame
neg_sent_df_filtered = neg_sent_df[mask]

# neg_sent_df = neg_sent_df[~neg_sent_df['sentence'].isin(false_negatives)]

In [22]:
len(neg_sent_df_filtered)

816

In [23]:
len(all_labelled_data)

639

# Clustering negative sentences

In [24]:
import altair as alt
import altair_saver
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
# from sklearn.manifold import TSNE
import umap

model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2024-07-03 11:09:29,695 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-07-03 11:09:30,057 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


In [25]:
def reduce_to_2D(vectors, random_state=1):
    """Helper function to reduce vectors to 2-d embeddings using UMAP, for visualisation purposes"""
    reducer = umap.UMAP(n_components=2, random_state=random_state)
    embedding = reducer.fit_transform(vectors)
    return embedding

In [26]:
embeddings = model.encode(neg_sent_df_filtered['sentence'].tolist())

embeddings_2d = reduce_to_2D(embeddings)

num_clusters = 50 #int(len(embeddings) ** 0.5)

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(embeddings_2d)

Batches: 100%|██████████| 26/26 [00:04<00:00,  5.48it/s]
/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
/Users/rosie.oxbury/miniconda3/envs/dap_job_quality/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

In [27]:
# assign the cluster names back into the dataframe
neg_sent_df_filtered['cluster'] = clusters

neg_sent_df_filtered['cluster'].value_counts()

neg_sent_df_filtered['x'] = embeddings_2d[:, 0]
neg_sent_df_filtered['y'] = embeddings_2d[:, 1]

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_33838/3140394937.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  neg_sent_df_filtered['cluster'] = clusters
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_33838/3140394937.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  neg_sent_df_filtered['x'] = embeddings_2d[:, 0]
/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_33838/3140394937.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Da

In [28]:
# Create the Altair chart
chart = alt.Chart(neg_sent_df_filtered).mark_circle(size=60).encode(
    x='x:Q',
    y='y:Q',
    color='cluster:N',
    tooltip=['cluster','sentence']
).properties(width=900, height=600).interactive()

# Display the chart
chart.show()

alt.Chart(...)

In [29]:
# Save the chart as a PNG file
chart.save('chart.html')

In [32]:
sample_size = round(len(all_labelled_data) / num_clusters)

In [33]:
neg_sent_df_sample = neg_sent_df_filtered.groupby('cluster', group_keys=False).apply(lambda x: x.sample(min(len(x), sample_size)))

/var/folders/x3/w5p1j0m5745bvc10jk_w36900000gn/T/ipykernel_33838/1479493853.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  neg_sent_df_sample = neg_sent_df_filtered.groupby('cluster', group_keys=False).apply(lambda x: x.sample(min(len(x), sample_size)))


In [34]:
len(neg_sent_df_sample)

634

In [43]:
neg_sent_df_sample['label'] = 0
neg_sent_df_sample['span_sents'] = None

In [38]:
all_labelled_data['label'] = 1

In [41]:
all_labelled_data.rename(columns={'true_sentence': 'sentence'}, inplace=True)
all_labelled_data

,id,text,label,sentences,span_sents,sentence
0,45332364,Occupational Therapy Technician Team Leader Se...,1,[Occupational Therapy Technician Team Leader S...,"£25,103.32pa","Salary £25,103.32pa."
1,45332364,Occupational Therapy Technician Team Leader Se...,1,[Occupational Therapy Technician Team Leader S...,Hours 38 per week,Hours 38 per week Benefits Extensive access...
2,45332364,Occupational Therapy Technician Team Leader Se...,1,[Occupational Therapy Technician Team Leader S...,Extensive access and support to recognised qua...,Hours 38 per week Benefits Extensive access...
3,45332364,Occupational Therapy Technician Team Leader Se...,1,[Occupational Therapy Technician Team Leader S...,Basic entitlement is 30.0 days (pro rata for h...,Basic entitlement is 30.0 days (pro rata for h...
4,45332364,Occupational Therapy Technician Team Leader Se...,1,[Occupational Therapy Technician Team Leader S...,This is inclusive of Bank Holidays.,This is inclusive of Bank Holidays.
...,...,...,...,...,...,...
634,46827704,Excellent Bonus Structure. Career Development ...,1,"[Excellent Bonus Structure., Career Developmen...",Excellent Bonus Structure.,Excellent Bonus Structure.
635,46827704,Excellent Bonus Structure. Career Development ...,1,"[Excellent Bonus Structure., Career Developmen...",Career Development Opportunities.,Career Development Opportunities.
636,46827704,Excellent Bonus Structure. Career Development ...,1,"[Excellent Bonus Structure., Career Developmen...",Company Discount,Company Discount.
637,46827704,Excellent Bonus Structure. Career Development ...,1,"[Excellent Bonus Structure., Career Developmen...",Childcare Vouchers,Childcare Vouchers


In [54]:
all_labelled_data.to_csv("positive_sents_for_labelling.csv", index=False)

In [47]:
all_data = pd.concat([neg_sent_df_sample[['id', 'sentence', 'label', 'span_sents']],all_labelled_data[['id', 'sentence', 'label', 'span_sents']]]).reset_index(drop=True)

In [48]:
all_data

,id,sentence,label,span_sents
0,45355569,"Management of the office, office-based staff a...",0,None
1,45275048,Following agreed processes to deliver accurate...,0,None
2,45467564,"Mechanical knowledge, previous experience in m...",0,None
3,45531352,The succesful Engineering Team Leader will be ...,0,None
4,45355569,Able to work under pressure and motivate other...,0,None
...,...,...,...,...
1268,46827704,Excellent Bonus Structure.,1,Excellent Bonus Structure.
1269,46827704,Career Development Opportunities.,1,Career Development Opportunities.
1270,46827704,Company Discount.,1,Company Discount
1271,46827704,Childcare Vouchers,1,Childcare Vouchers


In [51]:
from sklearn.model_selection import train_test_split

# Group by 'id' and calculate the proportion of each label within each 'id'
id_labels = all_data.groupby('id')['label'].apply(lambda x: x.value_counts(normalize=True)).unstack(fill_value=0)

# Add a column with the majority label for stratification
id_labels['majority_label'] = id_labels.idxmax(axis=1)

# Get unique ids and their majority label
ids_with_labels = id_labels.reset_index()[['id', 'majority_label']]

# Split the unique ids into training, validation, and test sets using stratified sampling
train_ids, temp_ids = train_test_split(ids_with_labels, test_size=0.3, stratify=ids_with_labels['majority_label'], random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, stratify=temp_ids['majority_label'], random_state=42)

In [52]:
# Create the training, validation, and test DataFrames
train_df = all_data[all_data['id'].isin(train_ids['id'])]
print(f"Train size: {len(train_df)}")
logging.info(train_df['label'].value_counts())
val_df = all_data[all_data['id'].isin(val_ids['id'])]
print(f"Val size: {len(val_df)}")
logging.info(val_df['label'].value_counts())
test_df = all_data[all_data['id'].isin(test_ids['id'])]
print(f"Test size: {len(test_df)}")
logging.info(test_df['label'].value_counts())

Train size: 907
2024-07-03 11:21:15,777 - root - INFO - label
0    461
1    446
Name: count, dtype: int64
Val size: 181
2024-07-03 11:21:15,779 - root - INFO - label
1    96
0    85
Name: count, dtype: int64
Test size: 185
2024-07-03 11:21:15,781 - root - INFO - label
1    97
0    88
Name: count, dtype: int64


In [56]:
positive_sents_manually_labelled = pd.read_csv(
        "s3://open-jobs-lake/job_quality/sentence_classifier/inputs/labelled/positive_sents_for_labelling - positive_sents_for_labelling.csv"
    )

In [72]:
category_representation = pd.DataFrame(positive_sents_manually_labelled['subcategory'].value_counts(normalize=True)).reset_index()
category_representation['perc'] = round(category_representation['proportion'] * 100)
category_representation

,subcategory,proportion,perc
0,COMP,0.251177,25.0
1,PERKS,0.171115,17.0
2,L&D,0.098901,10.0
3,HOURS,0.092622,9.0
4,LEAVE,0.061224,6.0
5,CAREER,0.058085,6.0
6,SOCIAL,0.051805,5.0
7,FLEX_LOC,0.045526,5.0
8,CONTRACT,0.043956,4.0
9,FLEX_HOURS,0.026688,3.0


In [73]:
from dap_job_quality.getters.keywords import get_keywords

keywords = get_keywords()

In [74]:
keywords

,dimension,subcategory,target_phrase,Notes
0,barriers to access,CARING,caring responsibilities,NaN
1,barriers to access,CARING,childcare vouchers,NaN
2,barriers to access,CARING,discount on childcare,NaN
3,barriers to access,CARING,parental leave,NaN
4,barriers to access,CARING,maternity,NaN
...,...,...,...,...
110,employment terms,LOC,field-based,NaN
111,work life balance,FLEX_LOC,hybrid,NaN
112,employment terms,LOC,office-based,NaN
113,work life balance,FLEX_LOC,remote,NaN


In [75]:
category_representation = pd.merge(category_representation, keywords[['dimension','subcategory']].drop_duplicates(), on='subcategory', how='outer')
category_representation

,subcategory,proportion,perc,dimension
0,AUTONOMY,0.001570,0.0,job design and nature of work
1,CAREER,0.058085,6.0,job design and nature of work
2,CARING,0.003140,0.0,barriers to access
3,COMP,0.251177,25.0,pay and benefits
4,CONTRACT,0.043956,4.0,employment terms
5,DISABILITY,NaN,NaN,barriers to access
6,FLEX_HOURS,0.026688,3.0,work life balance
7,FLEX_LOC,0.045526,5.0,work life balance
8,HEALTH,0.021978,2.0,"health, safety and psycho-social wellbeing"
9,HOURS,0.092622,9.0,employment terms


In [76]:
category_representation.to_csv("category_representation.csv", index=False)